In [2]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


# 0. Read the data

In [24]:
from config import PER_ORGANISM_PER_CLASS_ALLELES
from mhcgnomes import parse

DATA_PATH = "/home/denklewer/Data/schreiber_12_mers/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data_12_mers.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_12_mers_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_12_mers_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")

    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt

    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns


def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-Kd', 'H-2-Ld', 'H-2-Dd']

In [5]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

In [6]:
METHOD = 'mhcflurry'

## 0.1. make result dicts

In [7]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.2 Read decoy data

In [8]:
per_allele_decoy = dict()
for allele_name in ALLELES:
    organism =  defineOrganism(allele_name).lower()
    class_to_use = defineClass(allele_name)
    DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
    decoy_df =  pd.read_csv(DECOY_DATA_PATH)
    decoy_df['length'] = decoy_df['peptide'].str.len()
    
    per_allele_decoy[allele_name] = decoy_df.copy()

# 1. Predict with mhcflurry

In [9]:
# importing the predict module
from mhcflurry import Class1PresentationPredictor
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

In [10]:
from mhcgnomes import parse

# different allele naming.
TARGET_ALLELES = dict()

## allele normalization now with mhcgnomes
for allele_name in ALLELES:
    result = parse(allele_name)
    """
    parse(
        raw_name,
        only_class1=True,
        required_result_types=[Allele, AlleleWithoutGene, Gene],
        preferred_result_types=[Allele],
        use_allele_aliases=True,
        infer_class2_pairing=False,
        collapse_singleton_haplotypes=True,
        collapse_singleton_serotypes=True,
        raise_on_error=False)
    """
    target_allele_name = result.to_string()
    TARGET_ALLELES[allele_name] = target_allele_name


TARGET_ALLELES

{'H-2-Kd': 'H2-K*d', 'H-2-Ld': 'H2-L*d', 'H-2-Dd': 'H2-D*d'}

In [11]:
def filterByLength(per_allele_df, length_column='length'):
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    for allele_name in per_allele_df.keys():
        
        initial_len = len(per_allele_df[allele_name])
        if defineClass(allele_name) == 'I':
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 15]
        else:
            per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_df[allele_name][length_column] <= 30]
        result_len = len(per_allele_df[allele_name])
        print(f"Filtered out {initial_len - result_len} samples")
        

    
print(f"\t mt")
filterByLength(per_allele_mt_df)
print(f"\t wt")
filterByLength(per_allele_wt_df)

	 mt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
	 wt
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples


In [12]:
def predict_with_mhcflurry(per_allele_df , per_allele_ref_scores=None):
    per_allele_predictions = dict()
    
    predictor = Class1PresentationPredictor.load()
    supported_alleles =  predictor.supported_alleles
    
    for allele_name in ALLELES:
        df =  per_allele_df[allele_name].copy()
        target_allele = TARGET_ALLELES[allele_name]
        print(f"---{target_allele}")
        if target_allele in supported_alleles:

            peptides = df[df['peptide'].notnull()]['peptide'].values
            # affinity	best_allele	affinity_percentile	processing_score	presentation_score	presentation_percentile
            prediction_df = predictor.predict(alleles=[target_allele], peptides=peptides, include_affinity_percentile=True)
            result_df = df.copy()
            result_df.loc[result_df['peptide'].notnull(), 'affinity'] = prediction_df['affinity']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_ba'] = prediction_df['affinity_percentile']
            result_df.loc[result_df['peptide'].notnull(), 'processing_score'] = prediction_df['processing_score']
            result_df.loc[result_df['peptide'].notnull(), 'score_el'] = prediction_df['presentation_score']
            result_df.loc[result_df['peptide'].notnull(), 'percentile_rank_el'] = prediction_df['presentation_percentile']
            
            
            # calculate percentile ranks if mhcflurry doesn't have such scores for allele
            allele_to_percent_rank_transform = predictor.affinity_predictor.allele_to_percent_rank_transform
            if target_allele not in allele_to_percent_rank_transform:
                if per_allele_ref_scores:
                    print("Calculate percentile ranks")
                    scores  = 1 - np.log(result_df['affinity'])/np.log(50000)
                    ref_scores = 1 - np.log(per_allele_ref_scores[allele_name]['affinity'])/np.log(50000)
                    ref_scores = ref_scores.sort_values()
                    pct_increment = 1./len(ref_scores)
                    percentile_ranks = (0.5 + 0.5*ref_scores.searchsorted(scores, side='left') + 0.5*ref_scores.searchsorted(scores, side='right'))*pct_increment
                    percentile_ranks = 100 - percentile_ranks*100
                    result_df['percentile_rank_ba'] = percentile_ranks                
        else:
            #dummy prediction
            print("Method doesn't support {}".format(target_allele))
            result_df = df.copy()
            result_df['affinity'] = np.nan
            result_df['percentile_rank_ba'] = np.nan
            result_df['processing_score'] = np.nan
            result_df['score_el'] = np.nan
            result_df['percentile_rank_el'] = np.nan
        per_allele_predictions[allele_name] = result_df
        
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



## 0.2 Read decoy data

In [13]:
sample_predictor = Class1PresentationPredictor.load()

per_allele_decoy = dict()
for allele_name in ALLELES:
    target_allele = TARGET_ALLELES[allele_name]
    allele_to_percent_rank_transform = sample_predictor.affinity_predictor.allele_to_percent_rank_transform
    if target_allele not in allele_to_percent_rank_transform:
        organism =  defineOrganism(allele_name).lower()
        class_to_use = defineClass(allele_name)
        DECOY_DATA_PATH = f"/home/denklewer/Data/decoy/sampled_decoy_data_{organism}_class_{class_to_use}.csv"
        decoy_df =  pd.read_csv(DECOY_DATA_PATH)
        decoy_df['length'] = decoy_df['peptide'].str.len()

    per_allele_decoy[allele_name] = decoy_df.copy()

Forcing tensorflow backend.


Instructions for updating:
non-resource variables are not supported in the long term

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [14]:
%%time
print("decoy ")
filterByLength(per_allele_decoy)


decoy 
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
CPU times: user 41.7 ms, sys: 4.17 ms, total: 45.8 ms
Wall time: 42.6 ms


## First calculate scores for decoy

In [15]:
%%time
per_allele_decoy_predictions = predict_with_mhcflurry(per_allele_decoy)

  0%|          | 0/2 [00:00<?, ?it/s]

---H2-K*d
Predicting processing.


Instructions for updating:
The `validate_indices` argument has no effect. Indices are always validated on CPU and never validated on GPU.
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Mode

Predicting affinities.


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
100%|██████████| 1/1 [01:41<00:00, 101.48s/it]
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-K*d has no percentile rank information
  warnings.warn(msg)
  0%|          | 0/2 [00:00<?, ?it/s]

---H2-L*d
Predicting processing.


... (5 outputs left out)

Predicting affinities.


100%|██████████| 1/1 [01:07<00:00, 67.95s/it]


CPU times: user 16min 40s, sys: 1min 14s, total: 17min 54s
Wall time: 9min 51s


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-D*d has no percentile rank information
  warnings.warn(msg)


In [16]:
per_allele_decoy_predictions[ALLELES[0]]

,peptide,length,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,NRTYSFQR,8,31438.907770,NaN,0.240352,0.007504,31.643179
1,AFYSGHGA,8,17211.455841,NaN,0.399518,0.023283,10.231984
2,AKKQSLLD,8,32697.730681,NaN,0.028266,0.003425,99.286603
3,RWLVVNVAHDLL,12,24818.531625,NaN,0.196215,0.008072,27.617391
4,ARRGDLDVLRSL,12,22667.826575,NaN,0.119942,0.006739,37.359049
...,...,...,...,...,...,...,...
199995,PVINDILTLALS,12,31110.669026,NaN,0.012550,0.003400,99.286603
199996,QIFTGNVPGAG,11,26326.775713,NaN,0.030820,0.004262,62.744674
199997,PRESHDKN,8,31829.659909,NaN,0.000332,0.003185,99.286603
199998,IHSGAKPYSC,10,19724.129590,NaN,0.026339,0.005546,46.224674


In [17]:
%%time
print(f"--mt")
per_allele_mt_predictions = predict_with_mhcflurry(per_allele_mt_df, per_allele_decoy_predictions)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions  

print(f"--wt")
per_allele_wt_predictions = predict_with_mhcflurry(per_allele_wt_df, per_allele_decoy_predictions)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions  


--mt


  0%|          | 0/1 [00:00<?, ?it/s]

---H2-K*d
Predicting processing.


/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/tensorflow/python/keras/engine/training.py:2426: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automat

Predicting affinities.


100%|██████████| 1/1 [00:17<00:00, 17.86s/it]
/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-K*d has no percentile rank information
  warnings.warn(msg)
  0%|          | 0/1 [00:00<?, ?it/s]

... (20 outputs left out)

Predicting affinities.


100%|██████████| 1/1 [00:00<00:00,  1.15it/s]

Calculate percentile ranks
CPU times: user 1min 50s, sys: 1.46 s, total: 1min 51s
Wall time: 1min 40s



/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/mhcflurry/class1_affinity_predictor.py:1021: UserWarning: Allele H2-D*d has no percentile rank information
  warnings.warn(msg)


In [19]:
per_method_per_allele_wt_predictions[METHOD][ALLELES[0]]

,source_sequence_name,allele,peptide,length,affinity,percentile_rank_ba,processing_score,score_el,percentile_rank_el
0,seq0,H-2-Kd,FQDSLINQMTQI,12,24192.786458,28.37525,0.074636,0.005396,46.224674
1,seq1,H-2-Kd,QDSLINQMTQIK,12,29407.095002,70.86750,0.051928,0.004125,62.744674
2,seq2,H-2-Kd,DSLINQMTQIKL,12,29610.420439,73.11075,0.256764,0.008421,27.617391
3,seq3,H-2-Kd,SLINQMTQIKLS,12,30732.139345,85.47875,0.030164,0.003661,99.286603
4,seq4,H-2-Kd,LINQMTQIKLSV,12,31526.439995,92.78925,0.048251,0.003807,99.286603
...,...,...,...,...,...,...,...,...,...
2728,seq2728,H-2-Kd,ARYMPQNPHIIA,12,25964.231219,38.52925,0.548389,0.026362,9.155353
2729,seq2729,H-2-Kd,RYMPQNPHIIAT,12,9847.543613,4.85575,0.733759,0.117999,2.617337
2730,seq2730,H-2-Kd,YMPQNPHIIATK,12,23630.547348,25.92575,0.624764,0.037420,6.631304
2731,seq2731,H-2-Kd,MPQNPHIIATKT,12,30386.307480,81.79125,0.373676,0.012370,18.617989


# 2. Join predictions

In [20]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':['source_sequence_name', 'score', 'affinity', 'percentile_rank'], 
    'mhcflurry':['source_sequence_name', 'affinity', 'percentile_rank_ba', 'processing_score', 'score_el', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [21]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [22]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2-Kd
method mhcflurry
allele H-2-Ld
method mhcflurry
allele H-2-Dd
method mhcflurry


# 5. For now just write to files

In [25]:
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [26]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)

In [27]:
total_df

,source_sequence_name,allele,peptide_mt,length,mhcflurry_affinity_mt,mhcflurry_percentile_rank_ba_mt,mhcflurry_processing_score_mt,mhcflurry_score_el_mt,mhcflurry_percentile_rank_el_mt,peptide_wt,mhcflurry_affinity_wt,mhcflurry_percentile_rank_ba_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcflurry_percentile_rank_el_wt
0,seq0,H-2-Dd,FQDSLINQMTQS,12,28266.480579,71.72175,0.009194,0.003687,99.286603,FQDSLINQMTQI,24258.577173,36.42425,0.074636,0.005382,46.224674
1,seq1,H-2-Dd,QDSLINQMTQSK,12,26240.906999,51.40625,0.098935,0.005434,46.224674,QDSLINQMTQIK,26663.735087,55.29350,0.051928,0.004535,62.744674
2,seq2,H-2-Dd,DSLINQMTQSKL,12,27325.814578,61.69775,0.151514,0.006287,37.359049,DSLINQMTQIKL,27587.591350,64.46325,0.256764,0.009015,24.579239
3,seq3,H-2-Dd,SLINQMTQSKLS,12,29713.356582,87.66425,0.006641,0.003481,99.286603,SLINQMTQIKLS,29934.441205,89.80225,0.030164,0.003755,99.286603
4,seq4,H-2-Dd,LINQMTQSKLSV,12,29431.934335,84.76825,0.025851,0.003760,99.286603,LINQMTQIKLSV,30377.298389,93.51225,0.048251,0.003946,99.286603
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2728,seq2728,H-2-Dd,ARYMLQNPHIIA,12,26941.434503,57.92975,0.410557,0.015787,15.160951,ARYMPQNPHIIA,25046.531698,41.72325,0.548389,0.027274,8.839293
2729,seq2729,H-2-Dd,RYMLQNPHIIAT,12,30511.144417,94.45675,0.725526,0.041522,6.047908,RYMPQNPHIIAT,28258.394303,71.62875,0.733759,0.045848,5.554946
2730,seq2730,H-2-Dd,YMLQNPHIIATK,12,27306.882942,61.51475,0.500246,0.021281,11.115788,YMPQNPHIIATK,24664.826694,39.06275,0.624764,0.035950,6.962799
2731,seq2731,H-2-Dd,MLQNPHIIATKT,12,31047.696176,97.34925,0.041349,0.003771,99.286603,MPQNPHIIATKT,28960.939725,79.51625,0.373676,0.012953,18.617989
